In [3]:
import pandas as pd
from nba_api.stats.endpoints import synergyplaytypes
pd.set_option('display.max_columns', None)

In [4]:
pip install plotly

You should consider upgrading via the '/usr/local/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [5]:
data = synergyplaytypes.SynergyPlayTypes(
    league_id='00',
    season='2025-26',
    season_type_all_star='Playoffs',
    player_or_team_abbreviation='T',
    type_grouping_nullable='offensive',
    play_type_nullable='Isolation'
)

KeyboardInterrupt: 

In [ ]:
df = data.get_data_frames()[0]
df

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,PLAY_TYPE,TYPE_GROUPING,PERCENTILE,GP,POSS_PCT,PPP,FG_PCT,FT_POSS_PCT,TOV_POSS_PCT,SF_POSS_PCT,PLUSONE_POSS_PCT,SCORE_POSS_PCT,EFG_PCT,POSS,PTS,FGM,FGA,FGMX
0,42025,1610612739,CLE,Cleveland Cavaliers,Isolation,Offensive,0.867,9,0.113,0.965,0.404,0.133,0.097,0.115,0.018,0.434,0.466,113,109,36,89,53
1,42025,1610612750,MIN,Minnesota Timberwolves,Isolation,Offensive,0.933,9,0.099,0.990,0.434,0.167,0.069,0.147,0.049,0.471,0.458,102,101,36,83,47
2,42025,1610612755,PHI,Philadelphia 76ers,Isolation,Offensive,0.600,10,0.087,0.944,0.402,0.089,0.033,0.089,0.033,0.422,0.445,90,85,33,82,49
3,42025,1610612752,NYK,New York Knicks,Isolation,Offensive,1.000,9,0.078,1.079,0.477,0.158,0.053,0.158,0.066,0.500,0.492,76,82,31,65,34
4,42025,1610612759,SAS,San Antonio Spurs,Isolation,Offensive,0.800,8,0.093,0.963,0.433,0.122,0.085,0.122,0.024,0.451,0.478,82,79,29,67,38
5,42025,1610612738,BOS,Boston Celtics,Isolation,Offensive,0.467,7,0.098,0.919,0.393,0.108,0.135,0.095,0.000,0.405,0.455,74,68,22,56,34
6,42025,1610612760,OKC,Oklahoma City Thunder,Isolation,Offensive,0.667,6,0.101,0.955,0.389,0.152,0.045,0.121,0.015,0.455,0.435,66,63,21,54,33
7,42025,1610612765,DET,Detroit Pistons,Isolation,Offensive,0.000,9,0.086,0.714,0.355,0.155,0.119,0.143,0.012,0.393,0.363,84,60,22,62,40
8,42025,1610612753,ORL,Orlando Magic,Isolation,Offensive,0.067,7,0.092,0.718,0.308,0.183,0.141,0.183,0.056,0.338,0.356,71,51,16,52,36
9,42025,1610612747,LAL,Los Angeles Lakers,Isolation,Offensive,0.733,8,0.061,0.961,0.485,0.235,0.137,0.216,0.020,0.490,0.500,51,49,16,33,17


In [ ]:
def team_play_types_freq(play_types, team_df):

    for play_type, prefix in play_types.items():

        data = synergyplaytypes.SynergyPlayTypes(
            league_id='00',
            season='2025-26',
            season_type_all_star='Playoffs',
            player_or_team_abbreviation='T',
            type_grouping_nullable='offensive',
            play_type_nullable=play_type
        )

        df_play_type = data.get_data_frames()[0]

        df_play_type = df_play_type[[
            'TEAM_NAME',
            'POSS_PCT',
            'PPP',
            'PERCENTILE'
        ]]

        df_play_type = df_play_type.rename(columns={
            'POSS_PCT': f'{prefix}_FREQ',
            'PPP': f'{prefix}_PPP',
            'PERCENTILE': f'{prefix}_RK'
        })

        team_df = team_df.merge(
            df_play_type,
            on='TEAM_NAME',
            how='left'
        )

    freq_cols = [
        col for col in team_df.columns
        if col.endswith('_FREQ')
    ]

    team_df['TOT_FREQ'] = team_df[freq_cols].sum(axis=1)

    team_df['OTHER_FREQ'] = 1 - team_df['TOT_FREQ']

    return team_df

In [ ]:
play_types = {
    'Transition': 'TRANS',
    'Isolation': 'ISO',
    'PRBallHandler': 'PNR_BH',
    'PRRollman': 'PNR_RM',
    'Postup': 'POST',
    'Spotup' : 'SPOT',
    'OffScreen' :'OFF_SCREEN',
    'Handoff' : 'HANDOFF'
}

In [ ]:
team_df = pd.DataFrame({
    "TEAM_NAME": [
        "Boston Celtics",
        "Cleveland Cavaliers",
        "Detroit Pistons",
        "Oklahoma City Thunder"
    ]
})

In [ ]:
play_types_df = team_play_types_freq(play_types, team_df)
# freq_cols = [col for col in play_types_df.columns if col.endswith('_FREQ')]

#play_types_df['TOT_FREQ'] = play_types_df[freq_cols].sum(axis=1)
#play_types_df['OTHER_FREQ'] = 1 - play_types_df['TOT_FREQ']
play_types_df

,TEAM_NAME,TRANS_FREQ,TRANS_PPP,TRANS_RK,ISO_FREQ,ISO_PPP,ISO_RK,PNR_BH_FREQ,PNR_BH_PPP,PNR_BH_RK,PNR_RM_FREQ,PNR_RM_PPP,PNR_RM_RK,POST_FREQ,POST_PPP,POST_RK,SPOT_FREQ,SPOT_PPP,SPOT_RK,OFF_SCREEN_FREQ,OFF_SCREEN_PPP,OFF_SCREEN_RK,HANDOFF_FREQ,HANDOFF_PPP,HANDOFF_RK,TOT_FREQ,OTHER_FREQ
0,Boston Celtics,0.141,1.189,0.867,0.098,0.919,0.467,0.203,0.856,0.600,0.068,1.196,0.667,0.031,1.261,1.000,0.251,1.005,0.533,0.040,1.000,0.600,0.034,0.885,0.333,0.866,0.134
1,Cleveland Cavaliers,0.117,0.974,0.067,0.113,0.965,0.867,0.192,0.870,0.667,0.070,1.171,0.533,0.027,0.926,0.600,0.226,0.996,0.467,0.019,1.105,0.733,0.043,1.103,0.800,0.807,0.193
2,Detroit Pistons,0.160,1.121,0.467,0.086,0.714,0.000,0.186,0.896,0.733,0.049,0.958,0.200,0.043,0.833,0.333,0.187,0.962,0.333,0.038,0.757,0.200,0.040,1.200,1.000,0.789,0.211
3,Oklahoma City Thunder,0.167,1.211,0.933,0.101,0.955,0.667,0.158,1.165,1.000,0.044,1.207,0.733,0.018,1.167,0.933,0.288,1.096,0.800,0.000,1.286,0.933,0.047,1.038,0.733,0.823,0.177
